In [29]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import yaml

In [30]:
with open('../config.yaml', 'r') as file:
    config = yaml.safe_load(file)

In [31]:
# import the datasets
final_demo = pd.read_csv(config["input_data"]["file1"])
df_final_experiment_clients =pd.read_csv(config["input_data"]["file2"])
df_final_web_data_pt_1 = pd.read_csv(config["input_data"]["file3"])
df_final_web_data_pt_2 = pd.read_csv(config["input_data"]["file4"])

In [32]:
# check if there is an issue of column type
final_demo.dtypes, df_final_experiment_clients.dtypes, df_final_web_data_pt_1.dtypes

(client_id             int64
 clnt_tenure_yr      float64
 clnt_tenure_mnth    float64
 clnt_age            float64
 gendr                   str
 num_accts           float64
 bal                 float64
 calls_6_mnth        float64
 logons_6_mnth       float64
 dtype: object,
 client_id    int64
 Variation      str
 dtype: object,
 client_id       int64
 visitor_id        str
 visit_id          str
 process_step      str
 date_time         str
 dtype: object)

The dtype shows that the client_id is always an integer so theire is no issue for joining df

In [33]:
# remove rows from df_exeprimental_clients where variation is NA
df_final_experiment_clients = df_final_experiment_clients.dropna(subset=["Variation"])
df_final_experiment_clients.shape 

(50500, 2)

The decrease of number of rows matches the limitation to clients that participated

In [34]:
# combine the web data
concat_webdata = pd.concat([df_final_web_data_pt_1, df_final_web_data_pt_1], axis=0, join="outer")
concat_webdata.shape

(686282, 5)

In [35]:
# merge the demo
merged_demo = concat_webdata.merge(final_demo, on="client_id",how="left")
merged_demo.shape

(686282, 13)

The constant number of rows show the left join was made properly

In [36]:
# merge the experimental_clients
merged_experiment = merged_demo.merge(df_final_experiment_clients, on="client_id",how="left")
merged_experiment.shape

(686282, 14)

The constant number of rows show the left join was made properly

In [37]:
# separate process from final df
df = merged_experiment.copy()

In [38]:
# clean the duplicates values
df.drop_duplicates(inplace=True)

In [39]:
# keep clients involved in the AB Testing
df = df.dropna(subset=["Variation"])

In [40]:
# convert date str into a datetime
df['date_time'] = pd.to_datetime(df['date_time'])

In [44]:
# import the datasets
# df_main = df.to_csv(config["output_data"]["file5"])

Some client_ids were not matching. For example client_id = 6373617 was in df_final_web_data_pt_2 but not in expepriment either demo. This suggests a discrepancies between datasets (client_id in web data but not in demo for example)